# House prices — cleaning and preprocessing
My cleaning decisions distinguish structural absence, missing measurements and invalid labels. Some missing categories mean a feature is not present; I fill those with None. I retain rows with incomplete predictors and learn numeric medians and categorical modes from training folds.

For LotFrontage, zero does not represent a sensible unknown value. Median imputation is the default; my regression-imputation idea remains a possible CV experiment. GarageYrBlt is related to YearBuilt, so I retain it with a garage-age representation and inspect redundancy instead of applying a global correlation cutoff.

**My notes to edit:** describe each rule and document any exceptions found in the raw records.


## Setup and training-only data
All definitions are included here. This diagnostic notebook does not need outputs from exploration.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML

# Works when Jupyter starts in this folder, notebooks/, or the repository root.
locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations
     for candidate in (location, location / "regression" / "house_price_prediction")
     if (candidate / "house_prices_dataset" / "train" / "train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository or house_price_prediction folder.")
DATA_DIR = PROJECT_ROOT / "house_prices_dataset"
TRAIN_PATH = DATA_DIR / "train" / "train.csv"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET = "SalePrice"
ID_COLUMN = "Id"
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [ ]:
# Edit these settings here before Run All. No external configuration is read.
# Use None for every installed model, or enter registry names in a list:
# SELECTED_MODELS = ["DummyMedian", "Ridge", "GradientBoostedTreesModel"]
SELECTED_MODELS = None
config = {
    "random_state": 42,
    "ground_truth_fraction": 0.2,
    "validation_fraction": 0.2,
    "cv_folds": 5,
    "models": SELECTED_MODELS,
    "tune": True,
    "tune_top_n": 2,
    "overfit_ratio": 1.5,
    "underperformance_ratio": 1.25,
}
for key in ("ground_truth_fraction", "validation_fraction"):
    if not 0 < config[key] < 0.5:
        raise ValueError(f"{key} must be between 0 and 0.5")
if not isinstance(config["cv_folds"], int) or config["cv_folds"] < 2:
    raise ValueError("cv_folds must be an integer >= 2")
if not isinstance(config["tune"], bool) or not isinstance(config["tune_top_n"], int) or config["tune_top_n"] < 1:
    raise ValueError("tune must be boolean and tune_top_n must be a positive integer")
if SELECTED_MODELS is not None and (
    not isinstance(SELECTED_MODELS, list) or not SELECTED_MODELS
    or any(not isinstance(name, str) for name in SELECTED_MODELS)
    or len(SELECTED_MODELS) != len(set(SELECTED_MODELS))
):
    raise ValueError("SELECTED_MODELS must be None or a nonempty list of unique model names")
for key in ("overfit_ratio", "underperformance_ratio"):
    if not isinstance(config[key], (int, float)) or config[key] <= 1:
        raise ValueError(f"{key} must be a number greater than 1")
display(pd.Series(config, name="active_configuration").to_frame())


In [ ]:
from sklearn.model_selection import train_test_split

def validate_training_data(train: pd.DataFrame) -> dict:
    required = {TARGET}
    missing = required.difference(train.columns)
    if missing:
        raise ValueError(f"Training data is missing required columns: {sorted(missing)}")
    if train[TARGET].isna().any() or (train[TARGET] <= 0).any():
        raise ValueError("SalePrice must be present and strictly positive for log-target modelling.")
    if not np.isfinite(train[TARGET]).all() or train[TARGET].nunique() < 2:
        raise ValueError("Target must be finite and contain at least two distinct values.")
    if ID_COLUMN not in train or train[ID_COLUMN].isna().any() or train[ID_COLUMN].duplicated().any():
        raise ValueError("Training Id must be present, non-null and unique.")
    return {
        "rows": int(len(train)),
        "columns": int(train.shape[1]),
        "duplicate_rows": int(train.duplicated().sum()),
        "target_missing": int(train[TARGET].isna().sum()),
        "target_min": float(train[TARGET].min()),
        "target_max": float(train[TARGET].max()),
    }

def prepare_data(config=None, save=False):
    config = dict(config)
    raw = pd.read_csv(TRAIN_PATH)
    validate_training_data(raw)
    development, ground = train_test_split(raw, test_size=config["ground_truth_fraction"], random_state=config["random_state"])
    training, validation = train_test_split(development, test_size=config["validation_fraction"], random_state=config["random_state"])
    if config["cv_folds"] > len(training):
        raise ValueError("cv_folds exceeds training rows")
    if save:
        ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
        GROUND_TRUTH_DIR.mkdir(parents=True, exist_ok=True)
        training.to_csv(TRAIN_PATH.parent / "training_data.csv", index=False)
        validation.to_csv(TRAIN_PATH.parent / "validation_data.csv", index=False)
        ground.drop(columns=TARGET).to_csv(GROUND_TRUTH_DIR / "ground_truth_features.csv", index=False)
        ground[["Id", TARGET]].to_csv(GROUND_TRUTH_DIR / "ground_truth.csv", index=False)
    return training, validation, ground


In [ ]:
training, validation, ground = prepare_data(config)
print(validate_training_data(training))
assert not (set(training.Id) & set(validation.Id) or set(training.Id) & set(ground.Id) or set(validation.Id) & set(ground.Id))
print("Disjoint IDs confirmed.")
display(training.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_count"))


## Structural categories and ordinal mappings
I map absence in alley, fence, fireplace, garage, pool and basement categories before general imputation. Quality and finish values have a meaningful order; feature engineering adds ordinal representations while categories can still be encoded. Unknown future categories use the encoder's ignore behavior.

Id identifies records for comparison with actual prices and does not enter model fitting. MSSubClass is encoded as a category. **My notes:** inspect cases where a present component has an unknown quality rating; refine absence rules if needed.


In [ ]:
def engineer_features(X):
    """Apply row-wise domain rules; no statistics or labels are learned."""
    data = X.copy()
    if ID_COLUMN in data:
        data = data.drop(columns=[ID_COLUMN])

    # In the original analysis, these missing values were correctly
    # interpreted as "the feature is not present" rather than an unknown
    # category. Preserve that insight in the notebook feature function.
    structural_missing = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "GarageCond", "GarageQual", "GarageType", "GarageFinish", "BsmtFinType1", "BsmtFinType2", "BsmtQual", "BsmtCond", "BsmtExposure"]
    for column in structural_missing:
        if column in data:
            data[column] = data[column].fillna("None")

    def add_sum(name, columns):
        existing = [c for c in columns if c in data.columns]
        if existing:
            data[name] = data[existing].fillna(0).sum(axis=1)

    add_sum("TotalSF", ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"])
    add_sum("TotalBathrooms", ["FullBath", "BsmtFullBath"])
    if "HalfBath" in data and "BsmtHalfBath" in data:
        data["TotalBathrooms"] = data["TotalBathrooms"] + 0.5 * data[["HalfBath", "BsmtHalfBath"]].fillna(0).sum(axis=1)
    add_sum("TotalPorchSF", ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "WoodDeckSF"])
    add_sum("TotalQualityScore", ["OverallQual", "OverallCond"])

    # Preserve the original derived-feature vocabulary as aliases so the
    # earlier analysis remains traceable in model explanations.
    add_sum("PorchArea", ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"])
    if "YrSold" in data and "YearBuilt" in data:
        data["Time_Held"] = (data["YrSold"] - data["YearBuilt"]).clip(lower=0)
        data["House_Age"] = data["Time_Held"]
    if "TotalBsmtSF" in data and "BsmtUnfSF" in data:
        data["Finished_BasementArea"] = (data["TotalBsmtSF"] - data["BsmtUnfSF"]).clip(lower=0)
    if "1stFlrSF" in data and "2ndFlrSF" in data and "GrLivArea" in data:
        # GrLivArea already includes above-ground floors: avoid double counting.
        data["Total_Living_Area"] = data["GrLivArea"]
    add_sum("Full_Baths", ["BsmtFullBath", "FullBath"])
    if "HalfBath" in data and "BsmtHalfBath" in data:
        data["Half_Baths"] = data[["BsmtHalfBath", "HalfBath"]].fillna(0).sum(axis=1)

    ordinal_maps = {
        "ExterQual": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "ExterCond": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "HeatingQC": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "KitchenQual": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "CentralAir": {"N": 0, "Y": 1},
        "BsmtQual": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "BsmtCond": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "FireplaceQu": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "GarageQual": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "GarageCond": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "PoolQC": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "None": 0},
        "BsmtExposure": {"Gd": 3, "Av": 2, "Mn": 1, "No": 0, "None": 0},
        "BsmtFinType1": {"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0.5, "None": 0},
        "BsmtFinType2": {"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0.5, "None": 0},
    }
    for column, mapping in ordinal_maps.items():
        if column in data:
            data[f"{column}_Ordinal"] = data[column].map(mapping)

    if "YrSold" in data and "YearBuilt" in data:
        data["AgeAtSale"] = (data["YrSold"] - data["YearBuilt"]).clip(lower=0)
    if "YrSold" in data and "YearRemodAdd" in data:
        data["YearsSinceRemodel"] = (data["YrSold"] - data["YearRemodAdd"]).clip(lower=0)
    if "GarageYrBlt" in data and "YrSold" in data:
        data["GarageAgeAtSale"] = (data["YrSold"] - data["GarageYrBlt"]).clip(lower=0)

    for source, name in (
        ("GarageArea", "HasGarage"),
        ("TotalBsmtSF", "HasBasement"),
        ("Fireplaces", "HasFireplace"),
        ("PoolArea", "HasPool"),
        ("2ndFlrSF", "HasSecondFloor"),
    ):
        if source in data:
            data[name] = (data[source].fillna(0) > 0).astype(int)

    if "OverallQual" in data and "GrLivArea" in data:
        data["QualityLivableArea"] = data["OverallQual"] * data["GrLivArea"]
    if "OverallQual" in data and "TotalSF" in data:
        data["QualityTotalSF"] = data["OverallQual"] * data["TotalSF"]
    if "ExterQual_Ordinal" in data and "ExterCond_Ordinal" in data:
        data["Exterior_Quality"] = data["ExterQual_Ordinal"] + data["ExterCond_Ordinal"]
    if "GarageQual_Ordinal" in data and "GarageCond_Ordinal" in data:
        data["Garage_Quality"] = data["GarageQual_Ordinal"] + data["GarageCond_Ordinal"]
    if "OverallQual" in data and "OverallCond" in data:
        data["Average_Quality"] = data["OverallQual"] + data["OverallCond"]
    # Use one canonical representation for exact aliases.
    data = data.drop(columns=["Time_Held", "House_Age", "Total_Living_Area", "TotalQualityScore"], errors="ignore")
    if "MSSubClass" in data:
        data["MSSubClass"] = data["MSSubClass"].astype(str)
    return data


In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.dummy import DummyRegressor

def make_preprocessor(X: pd.DataFrame, scale_numeric: bool = True) -> ColumnTransformer:
    numeric = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical = X.select_dtypes(exclude=[np.number]).columns.tolist()
    numeric_steps = [("impute", SimpleImputer(strategy="median", keep_empty_features=True))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn < 1.2
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            ("categorical", Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                ("onehot", encoder),
            ]), categorical),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

def price_model(X, estimator):
    # sklearn's estimator composition keeps learned preprocessing inside CV.
    # X already contains stateless engineered features, with Id removed.
    trees = (GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor)
    regressor = Pipeline([
        ("preprocess", make_preprocessor(X, scale_numeric=not isinstance(estimator, trees))),
        ("model", estimator),
    ])
    return TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)


In [ ]:
X = engineer_features(training.drop(columns=TARGET))
display(X[["Alley", "FireplaceQu", "GarageType", "BsmtQual"]].head(10))
preprocessor = make_preprocessor(X)
transformed = preprocessor.fit_transform(X)
print("Preprocessed matrix shape:", transformed.shape)
assert np.isfinite(transformed).all()
print("All model inputs are finite. This is a diagnostic fit on training data only.")


## Scaling, encoding and outlier decisions
Linear models receive standardized numeric inputs; tree models use imputed inputs without requiring standardization. One-hot encoding is fitted inside each training fold so validation categories do not influence fitting. My earlier row dropping and upper-tail filtering ideas need validation; the current default keeps those rows and uses stable transformations.

**My notes:** describe residual data-quality concerns and propose a controlled experiment for robust imputation or outlier handling.
